# AI Web Search (v1)

A minimal RAG-based AI web search demo.

**Workflow:**
1. User submits a query
2. **Web Search** — Tavily fetches the top 10 results (title, URL, snippet)
3. **LLM Generation** — a single LiteLLM call produces two sections:
   - **Answer**: synthesized response with inline citations linking to source URLs
   - **Web Search Results**: all 10 results listed with clickable titles and snippets
4. **Display** — markdown output is converted to HTML and rendered in Colab

In [ ]:
%pip install tavily-python litellm markdown -q

## Configuration

Set your API keys and choose a model before running.

- **TAVILY_API_KEY** — get one at [tavily.com](https://tavily.com)
- **OPENAI_API_KEY** — get one at [platform.openai.com](https://platform.openai.com)
- **MODEL** — any LiteLLM-supported model string; defaults to `gpt-4o-mini` (fast and cheap)

In [ ]:
import os

# Set API keys before running
os.environ["TAVILY_API_KEY"] = "tvly-..."   # get from tavily.com
os.environ["OPENAI_API_KEY"] = "sk-..."     # get from platform.openai.com

MODEL = "gpt-4o-mini"  # selectable: e.g. "gpt-4o", "claude-3-haiku-20240307"
TOP_K = 10             # number of web search results to retrieve

## Step 1 — Web Search

`search_web` calls the Tavily API and returns the top K results.
Each result contains `title`, `url`, and `content` (a snippet of the page text).

In [ ]:
from tavily import TavilyClient

def search_web(query: str, k: int = TOP_K) -> list:
    # Search the web via Tavily; return top k results
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    response = client.search(query, max_results=k)
    return response["results"]  # each item: {title, url, content, score, ...}

## Step 2 — LLM Generation

`build_prompt` formats the query and all search results into a single prompt.

`generate_answer` sends that prompt to the LLM via LiteLLM and returns markdown text
with two sections:
- `## Answer` — uses only relevant results; adds `[Title](URL)` citations inline
- `## Web Search Results` — lists every result with a clickable title and snippet

In [ ]:
import litellm

def build_prompt(query: str, results: list) -> str:
    # Format query + numbered search results into a single LLM prompt
    results_text = ""
    for i, r in enumerate(results, 1):
        results_text += (
            f"\n[{i}] Title: {r['title']}\n"
            f"    URL: {r['url']}\n"
            f"    Snippet: {r['content']}\n"
        )
    n = len(results)
    instructions = (
        "Instructions:\n"
        "1. Write an ## Answer section answering the query using ONLY information from the results.\n"
        "   - Ignore results that are irrelevant to the query.\n"
        "   - Add inline citations as markdown links [Title](URL) near supporting text.\n"
        f"2. Write a ## Web Search Results section listing ALL {n} results as:\n"
        "   ### [Title](URL)\n"
        "   Snippet text\n"
        "\nOutput only valid markdown."
    )
    return f"User Query: {query}\n\nWeb Search Results:\n{results_text}\n{instructions}"


def generate_answer(query: str, results: list, model: str = MODEL) -> str:
    # Call the LLM via LiteLLM; return the markdown response
    system = "You are a helpful research assistant. Answer accurately and cite sources."
    response = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": build_prompt(query, results)},
        ],
    )
    return response.choices[0].message.content

## Step 3 — Render Output

`run` orchestrates the full pipeline and renders the result as HTML in Colab.
The `markdown` library converts the LLM's markdown text to HTML;
`IPython.display.HTML` renders it inline in the notebook.

In [ ]:
import markdown
from IPython.display import HTML, display

def run(query: str, model: str = MODEL) -> None:
    # End-to-end: search -> generate -> render HTML
    print(f"Searching: {query!r} ...")
    results = search_web(query)
    print(f"Got {len(results)} results. Generating answer with {model} ...")
    md_output = generate_answer(query, results, model)
    # Convert markdown to HTML and display inline in Colab
    html_output = markdown.markdown(md_output, extensions=["extra"])
    display(HTML(html_output))

In [ ]:
run("What are the latest AI breakthroughs in 2025?")